In [1]:
%pip install mysql-connector-python pandas

     ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
     --------------------------------------- 0.0/17.7 MB 660.6 kB/s eta 0:00:27
     ---------------------------------------- 0.2/17.7 MB 1.7 MB/s eta 0:00:11
      --------------------------------------- 0.3/17.7 MB 2.6 MB/s eta 0:00:07
     - -------------------------------------- 0.6/17.7 MB 3.2 MB/s eta 0:00:06
     - -------------------------------------- 0.8/17.7 MB 3.5 MB/s eta 0:00:05
     -- ------------------------------------- 1.1/17.7 MB 3.9 MB/s eta 0:00:05
     --- ------------------------------------ 1.4/17.7 MB 4.1 MB/s eta 0:00:04
     --- ------------------------------------ 1.7/17.7 MB 4.5 MB/s eta 0:00:04
     ---- ----------------------------------- 2.0/17.7 MB 4.7 MB/s eta 0:00:04
     ---- ----------------------------------- 2.1/17.7 MB 4.8 MB/s eta 0:00:04
     ---- ----------------------------------- 2.1/17.7 MB 4.8 MB/


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mysql.connector
import pandas as pd
from IPython.display import display

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="A@b@1234"
)
cursor = connection.cursor()
cursor.execute("SELECT VERSION()")
mysql_version = tuple(int(part) for part in cursor.fetchone()[0].split("-")[0].split(".")[:3])
if mysql_version < (8, 0, 31):
    raise RuntimeError("MySQL 8.0.31 or newer is required.")
cursor.execute("CREATE DATABASE IF NOT EXISTS week1_sql_practical")
cursor.execute("USE week1_sql_practical")
connection.commit()

def show(sql):
    cursor.execute(sql)
    display(pd.DataFrame(cursor.fetchall(), columns=[column[0] for column in cursor.description]))


In [4]:
cursor.execute("DROP TABLE IF EXISTS course_batches")
cursor.execute("""
CREATE TABLE course_batches (
    batch_id INT PRIMARY KEY,
    subject_id VARCHAR(5) NOT NULL,
    batch_name VARCHAR(40) NOT NULL,
    start_date DATE NOT NULL,
    capacity INT CHECK (capacity > 0),
    trainer VARCHAR(40) NULL,
    status VARCHAR(10) DEFAULT 'Planned'
) ENGINE=InnoDB
""")
cursor.executemany(
    """INSERT INTO course_batches
    (batch_id, subject_id, batch_name, start_date, capacity, trainer)
    VALUES (%s, %s, %s, %s, %s, %s)""",
    [
        (1, "S01", "SQL Morning Batch", "2026-04-01", 30, None),
        (2, "S02", "Python Weekend Batch", "2026-04-05", 25, "Ritu Sharma"),
        (3, "S03", "Visualization Evening", "2026-05-10", 20, None),
        (4, "S01", "SQL Evening Batch", "2026-03-20", 30, "Amit Bansal")
    ]
)
connection.commit()
try:
    cursor.execute(
        """INSERT INTO course_batches
        (batch_id, subject_id, batch_name, start_date, capacity, trainer)
        VALUES (5, 'S04', 'Stats Crash Course', '2026-04-15', 0, NULL)"""
    )
except mysql.connector.Error as err:
    if err.errno != 3819:
        raise
else:
    raise AssertionError("The capacity CHECK constraint did not reject capacity 0.")
cursor.execute("UPDATE course_batches SET trainer = 'Neha Joshi' WHERE batch_id = 1")
cursor.execute("UPDATE course_batches SET trainer = 'Ritu Sharma' WHERE batch_id = 3")
connection.commit()
connection.start_transaction()
cursor.execute("DELETE FROM course_batches WHERE status = 'Planned'")
connection.rollback()
cursor.execute("UPDATE course_batches SET status = 'Open' WHERE start_date <= '2026-04-01'")
connection.commit()
show("SELECT batch_id, subject_id, batch_name, start_date, capacity, trainer, status FROM course_batches ORDER BY batch_id")

,batch_id,subject_id,batch_name,start_date,capacity,trainer,status
0,1,S01,SQL Morning Batch,2026-04-01,30,Neha Joshi,Open
1,2,S02,Python Weekend Batch,2026-04-05,25,Ritu Sharma,Planned
2,3,S03,Visualization Evening,2026-05-10,20,Ritu Sharma,Planned
3,4,S01,SQL Evening Batch,2026-03-20,30,Amit Bansal,Open


In [5]:
cursor.execute("DROP TABLE IF EXISTS students")
cursor.execute("""
CREATE TABLE students (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(60) NOT NULL,
    gender CHAR(1),
    city VARCHAR(50),
    email VARCHAR(100),
    phone VARCHAR(15),
    dob DATE,
    join_date DATE
) ENGINE=InnoDB
""")
cursor.executemany(
    """INSERT INTO students
    (student_id, student_name, gender, city, email, phone, dob, join_date)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
    [
        (101, "aarav sharma", "M", "Jaipur", "aarav.sharma@gmail.com", "9876543210", "2004-05-12", "2026-01-05"),
        (102, "Diya Patel", "F", "Ahmedabad", "diya.patel@yahoo.com", None, "2003-11-23", "2026-01-10"),
        (103, "Kabir Singh", "M", "Delhi", None, "9812345678", "2004-02-02", "2026-01-18"),
        (104, "Meera Iyer", "F", "Chennai", "meera.iyer@outlook.com", "9900112233", None, "2026-02-03"),
        (105, "rohan VERMA", "M", "Jaipur", "rohan.verma@gmail.com", "9811122233", "2005-07-30", "2026-02-14"),
        (106, "Sana Khan", "F", "Delhi", None, None, None, "2026-02-20"),
        (107, "Ishaan Gupta", "M", "Pune", "ishaan.gupta@gmail.com", "9822001100", "2004-09-09", "2026-03-01"),
        (108, "Ananya Rao", "F", None, None, None, "2003-12-19", "2026-03-05"),
        (109, "Vikram Joshi", "M", "Jaipur", "vikram.joshi@yahoo.com", "9876501234", "2004-01-25", "2026-03-11"),
        (110, "Neha Kapoor", "F", "Mumbai", "neha.kapoor@gmail.com", "9800098000", "2005-03-14", "2026-03-20")
    ]
)
connection.commit()
show("""
SELECT student_id, student_name, city, join_date
FROM students
WHERE city IN ('Jaipur', 'Delhi')
  AND join_date BETWEEN '2026-02-01' AND '2026-03-31'
ORDER BY join_date DESC
""")
show("""
SELECT DISTINCT city
FROM students
WHERE city IS NOT NULL
ORDER BY city
""")


,student_id,student_name,city,join_date
0,109,Vikram Joshi,Jaipur,2026-03-11
1,106,Sana Khan,Delhi,2026-02-20
2,105,rohan VERMA,Jaipur,2026-02-14


,city
0,Ahmedabad
1,Chennai
2,Delhi
3,Jaipur
4,Mumbai
5,Pune


In [6]:
show("""
SELECT
    student_id,
    UPPER(TRIM(student_name)) AS clean_name,
    CASE
        WHEN email IS NULL THEN 'Not Provided'
        ELSE SUBSTRING_INDEX(email, '@', 1)
    END AS email_user,
    CASE
        WHEN email IS NULL THEN 'Not Provided'
        ELSE SUBSTRING_INDEX(email, '@', -1)
    END AS email_domain,
    CASE
        WHEN phone IS NULL THEN 'Not Provided'
        ELSE CONCAT('XXXXXX', RIGHT(phone, 4))
    END AS masked_phone,
    MONTHNAME(join_date) AS join_month,
    DATEDIFF('2026-10-01', join_date) AS days_with_us
FROM students
ORDER BY student_id
""")


,student_id,clean_name,email_user,email_domain,masked_phone,join_month,days_with_us
0,101,AARAV SHARMA,aarav.sharma,gmail.com,XXXXXX3210,January,269
1,102,DIYA PATEL,diya.patel,yahoo.com,Not Provided,January,264
2,103,KABIR SINGH,Not Provided,Not Provided,XXXXXX5678,January,256
3,104,MEERA IYER,meera.iyer,outlook.com,XXXXXX2233,February,240
4,105,ROHAN VERMA,rohan.verma,gmail.com,XXXXXX2233,February,229
5,106,SANA KHAN,Not Provided,Not Provided,Not Provided,February,223
6,107,ISHAAN GUPTA,ishaan.gupta,gmail.com,XXXXXX1100,March,214
7,108,ANANYA RAO,Not Provided,Not Provided,Not Provided,March,210
8,109,VIKRAM JOSHI,vikram.joshi,yahoo.com,XXXXXX1234,March,204
9,110,NEHA KAPOOR,neha.kapoor,gmail.com,XXXXXX8000,March,195


In [7]:
cursor.execute("DROP TABLE IF EXISTS fee_payments")
cursor.execute("""
CREATE TABLE fee_payments (
    payment_id VARCHAR(5) PRIMARY KEY,
    student_id INT NOT NULL,
    subject_id VARCHAR(5) NOT NULL,
    payment_date DATE NOT NULL,
    amount DECIMAL(10, 2),
    status VARCHAR(20) NOT NULL
) ENGINE=InnoDB
""")
cursor.executemany(
    """INSERT INTO fee_payments
    (payment_id, student_id, subject_id, payment_date, amount, status)
    VALUES (%s, %s, %s, %s, %s, %s)""",
    [
        ("P01", 101, "S01", "2026-01-06", 4000, "Success"),
        ("P02", 101, "S02", "2026-01-20", 5000, "Success"),
        ("P03", 102, "S01", "2026-01-11", 4000, "Failed"),
        ("P04", 102, "S01", "2026-01-12", 4000, "Success"),
        ("P05", 103, "S02", "2026-01-19", 2500, "Success"),
        ("P06", 103, "S03", "2026-02-01", 3500, "Success"),
        ("P07", 104, "S01", "2026-02-04", 4000, "Success"),
        ("P08", 105, "S02", "2026-02-15", 5000, "Success"),
        ("P09", 105, "S04", "2026-02-16", None, "Pending"),
        ("P10", 103, "S02", "2026-02-19", 2500, "Success"),
        ("P11", 107, "S01", "2026-03-02", 4000, "Success"),
        ("P12", 107, "S02", "2026-03-03", 5000, "Refunded"),
        ("P13", 108, "S03", "2026-03-06", 3500, "Success"),
        ("P14", 109, "S01", "2026-03-12", 4000, "Success"),
        ("P15", 110, "S05", "2026-03-21", 4500, "Success")
    ]
)
connection.commit()
show("""
SELECT
    student_id,
    student_name,
    (email IS NULL) + (phone IS NULL) + (city IS NULL) + (dob IS NULL) AS missing_fields,
    CASE
        WHEN (email IS NULL) + (phone IS NULL) + (city IS NULL) + (dob IS NULL) = 0 THEN 'Complete'
        WHEN email IS NULL AND phone IS NULL THEN 'Contact Missing'
        ELSE 'Partially Complete'
    END AS profile_status
FROM students
ORDER BY student_id
""")
show("""
SELECT
    payment_id,
    student_id,
    amount,
    status,
    CASE
        WHEN amount IS NULL THEN 'Amount Missing'
        WHEN status <> 'Success' THEN 'Not Collected'
        ELSE 'Collected'
    END AS payment_class
FROM fee_payments
ORDER BY payment_id
""")
show("""
SELECT
    ROUND(AVG(amount), 2) AS avg_amount_correct,
    ROUND(AVG(COALESCE(amount, 0)), 2) AS avg_amount_wrong
FROM fee_payments
""")


,student_id,student_name,missing_fields,profile_status
0,101,aarav sharma,0,Complete
1,102,Diya Patel,1,Partially Complete
2,103,Kabir Singh,1,Partially Complete
3,104,Meera Iyer,1,Partially Complete
4,105,rohan VERMA,0,Complete
5,106,Sana Khan,3,Contact Missing
6,107,Ishaan Gupta,0,Complete
7,108,Ananya Rao,3,Contact Missing
8,109,Vikram Joshi,0,Complete
9,110,Neha Kapoor,0,Complete


,payment_id,student_id,amount,status,payment_class
0,P01,101,4000.00,Success,Collected
1,P02,101,5000.00,Success,Collected
2,P03,102,4000.00,Failed,Not Collected
3,P04,102,4000.00,Success,Collected
4,P05,103,2500.00,Success,Collected
5,P06,103,3500.00,Success,Collected
6,P07,104,4000.00,Success,Collected
7,P08,105,5000.00,Success,Collected
8,P09,105,None,Pending,Amount Missing
9,P10,103,2500.00,Success,Collected


,avg_amount_correct,avg_amount_wrong
0,3964.29,3700.00


In [8]:
cursor.execute("DROP TABLE IF EXISTS enrollments")
cursor.execute("DROP TABLE IF EXISTS subjects")
cursor.execute("""
CREATE TABLE subjects (
    subject_id VARCHAR(5) PRIMARY KEY,
    subject_name VARCHAR(60) NOT NULL,
    fee DECIMAL(10, 2) NOT NULL,
    max_seats INT NOT NULL
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE enrollments (
    enrollment_id VARCHAR(5) PRIMARY KEY,
    student_id INT NOT NULL,
    subject_id VARCHAR(5) NOT NULL,
    enroll_date DATE NOT NULL
) ENGINE=InnoDB
""")
cursor.executemany(
    "INSERT INTO subjects (subject_id, subject_name, fee, max_seats) VALUES (%s, %s, %s, %s)",
    [
        ("S01", "SQL Fundamentals", 4000, 30),
        ("S02", "Python Programming", 5000, 30),
        ("S03", "Data Visualization", 3500, 20),
        ("S04", "Statistics", 3000, 25),
        ("S05", "Cloud Basics", 4500, 20),
        ("S06", "Machine Learning", 6000, 15)
    ]
)
cursor.executemany(
    "INSERT INTO enrollments (enrollment_id, student_id, subject_id, enroll_date) VALUES (%s, %s, %s, %s)",
    [
        ("E01", 101, "S01", "2026-01-06"),
        ("E02", 101, "S02", "2026-01-06"),
        ("E03", 102, "S01", "2026-01-11"),
        ("E04", 103, "S02", "2026-01-19"),
        ("E05", 103, "S03", "2026-02-01"),
        ("E06", 104, "S01", "2026-02-04"),
        ("E07", 105, "S02", "2026-02-15"),
        ("E08", 105, "S04", "2026-02-15"),
        ("E09", 107, "S01", "2026-03-02"),
        ("E10", 107, "S02", "2026-03-02"),
        ("E11", 108, "S03", "2026-03-06"),
        ("E12", 109, "S01", "2026-03-12"),
        ("E13", 110, "S05", "2026-03-21"),
        ("E14", 112, "S02", "2026-03-25")
    ]
)
connection.commit()
show("""
SELECT
    s.subject_id,
    s.subject_name,
    COALESCE(e.enrolled_students, 0) AS enrolled_students,
    s.max_seats - COALESCE(e.enrolled_students, 0) AS seats_left,
    CASE
        WHEN COALESCE(e.enrolled_students, 0) >= 4 THEN 'High'
        WHEN COALESCE(e.enrolled_students, 0) BETWEEN 2 AND 3 THEN 'Medium'
        WHEN COALESCE(e.enrolled_students, 0) = 1 THEN 'Low'
        ELSE 'No Demand'
    END AS demand_level
FROM subjects s
LEFT JOIN (
    SELECT
        en.subject_id,
        COUNT(DISTINCT en.student_id) AS enrolled_students
    FROM enrollments en
    INNER JOIN students st ON st.student_id = en.student_id
    GROUP BY en.subject_id
) e ON e.subject_id = s.subject_id
ORDER BY enrolled_students DESC, s.subject_id
""")


,subject_id,subject_name,enrolled_students,seats_left,demand_level
0,S01,SQL Fundamentals,5,25,High
1,S02,Python Programming,4,26,High
2,S03,Data Visualization,2,18,Medium
3,S04,Statistics,1,24,Low
4,S05,Cloud Basics,1,19,Low
5,S06,Machine Learning,0,15,No Demand


In [9]:
show("""
SELECT
    DATE_FORMAT(payment_date, '%Y-%m') AS pay_month,
    SUM(CASE WHEN status = 'Success' THEN 1 ELSE 0 END) AS successful_payments,
    SUM(CASE WHEN status = 'Success' THEN amount ELSE 0 END) AS total_revenue,
    ROUND(AVG(CASE WHEN status = 'Success' THEN amount END), 0) AS avg_payment,
    SUM(CASE WHEN status <> 'Success' THEN 1 ELSE 0 END) AS other_payments
FROM fee_payments
GROUP BY DATE_FORMAT(payment_date, '%Y-%m')
HAVING SUM(CASE WHEN status = 'Success' THEN amount ELSE 0 END) >= 15000
ORDER BY pay_month
""")
show("""
SELECT
    s.subject_name,
    SUM(fp.amount) AS total_revenue
FROM subjects s
INNER JOIN fee_payments fp ON fp.subject_id = s.subject_id
WHERE fp.status = 'Success'
GROUP BY s.subject_id, s.subject_name
HAVING SUM(fp.amount) > 8000
ORDER BY total_revenue DESC
""")


,pay_month,successful_payments,total_revenue,avg_payment,other_payments
0,2026-01,4,15500.00,3875,1
1,2026-02,4,15000.00,3750,1
2,2026-03,4,16000.00,4000,1


,subject_name,total_revenue
0,SQL Fundamentals,20000.00
1,Python Programming,15000.00


In [10]:
cursor.execute("DROP TABLE IF EXISTS store_orders")
cursor.execute("DROP TABLE IF EXISTS store_customers")
cursor.execute("""
CREATE TABLE store_customers (
    customer_id VARCHAR(5) PRIMARY KEY,
    customer_name VARCHAR(60) NOT NULL,
    city VARCHAR(50)
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE store_orders (
    order_id VARCHAR(10) PRIMARY KEY,
    customer_id VARCHAR(5),
    order_date DATE NOT NULL,
    amount DECIMAL(10, 2) NOT NULL
) ENGINE=InnoDB
""")
cursor.executemany(
    "INSERT INTO store_customers (customer_id, customer_name, city) VALUES (%s, %s, %s)",
    [
        ("C01", "Rahul Mehta", "Jaipur"),
        ("C02", "Priya Nair", "Kochi"),
        ("C03", "Arjun Das", "Kolkata"),
        ("C04", "Fatima Sheikh", "Mumbai"),
        ("C05", "Karan Malhotra", "Delhi")
    ]
)
cursor.executemany(
    "INSERT INTO store_orders (order_id, customer_id, order_date, amount) VALUES (%s, %s, %s, %s)",
    [
        ("O1001", "C01", "2026-03-01", 1200),
        ("O1002", "C02", "2026-03-02", 850),
        ("O1003", "C01", "2026-03-05", 430),
        ("O1004", "C07", "2026-03-06", 999),
        ("O1005", "C04", "2026-03-08", 2100),
        ("O1006", None, "2026-03-09", 300)
    ]
)
connection.commit()
show("""
SELECT s.student_id, s.student_name, s.city
FROM students s
WHERE NOT EXISTS (
    SELECT 1
    FROM enrollments e
    WHERE e.student_id = s.student_id
)
ORDER BY s.student_id
""")
show("""
SELECT c.customer_id, c.customer_name, c.city
FROM store_customers c
LEFT JOIN store_orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id
""")
show("""
SELECT
    o.order_id,
    o.customer_id,
    o.amount,
    CASE
        WHEN o.customer_id IS NULL THEN 'Guest order - no customer'
        ELSE 'Customer not found'
    END AS issue
FROM store_orders o
LEFT JOIN store_customers c ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL OR c.customer_id IS NULL
ORDER BY o.order_id
""")


,student_id,student_name,city
0,106,Sana Khan,Delhi


,customer_id,customer_name,city
0,C03,Arjun Das,Kolkata
1,C05,Karan Malhotra,Delhi


,order_id,customer_id,amount,issue
0,O1004,C07,999.00,Customer not found
1,O1006,None,300.00,Guest order - no customer


In [11]:
cursor.execute("DROP TABLE IF EXISTS payments")
cursor.execute("DROP TABLE IF EXISTS order_items")
cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("""
CREATE TABLE customers (
    customer_id VARCHAR(5) PRIMARY KEY,
    customer_name VARCHAR(60) NOT NULL,
    city VARCHAR(50),
    signup_date DATE
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE products (
    product_id VARCHAR(5) PRIMARY KEY,
    product_name VARCHAR(60) NOT NULL,
    category VARCHAR(40),
    unit_price DECIMAL(10, 2) NOT NULL
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id VARCHAR(5) NOT NULL,
    order_date DATE NOT NULL,
    status VARCHAR(20) NOT NULL
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE order_items (
    order_id INT NOT NULL,
    product_id VARCHAR(5) NOT NULL,
    qty INT NOT NULL,
    unit_price DECIMAL(10, 2) NOT NULL,
    PRIMARY KEY (order_id, product_id)
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE payments (
    payment_id VARCHAR(5) PRIMARY KEY,
    order_id INT NOT NULL,
    payment_date DATE NOT NULL,
    amount DECIMAL(10, 2) NOT NULL,
    method VARCHAR(20) NOT NULL
) ENGINE=InnoDB
""")
cursor.executemany(
    "INSERT INTO customers (customer_id, customer_name, city, signup_date) VALUES (%s, %s, %s, %s)",
    [
        ("C01", "Rahul Mehta", "Jaipur", "2025-11-02"),
        ("C02", "Priya Nair", "Kochi", "2025-12-15"),
        ("C03", "Arjun Das", "Kolkata", "2026-01-08"),
        ("C04", "Fatima Sheikh", "Mumbai", "2026-01-20"),
        ("C05", "Karan Malhotra", "Delhi", "2026-02-11"),
        ("C06", "Sneha Kulkarni", "Pune", "2026-02-25")
    ]
)
cursor.executemany(
    "INSERT INTO products (product_id, product_name, category, unit_price) VALUES (%s, %s, %s, %s)",
    [
        ("P1", "Wireless Mouse", "Electronics", 799),
        ("P2", "USB-C Hub", "Electronics", 1999),
        ("P3", "Notebook A5", "Stationery", 120),
        ("P4", "Gel Pen Pack", "Stationery", 250),
        ("P5", "Desk Lamp", "Home", 1499),
        ("P6", "Water Bottle", "Home", 499)
    ]
)
cursor.executemany(
    "INSERT INTO orders (order_id, customer_id, order_date, status) VALUES (%s, %s, %s, %s)",
    [
        (1001, "C01", "2026-03-02", "Delivered"),
        (1002, "C02", "2026-03-03", "Delivered"),
        (1003, "C01", "2026-03-10", "Delivered"),
        (1004, "C03", "2026-03-12", "Cancelled"),
        (1005, "C04", "2026-03-15", "Shipped"),
        (1006, "C05", "2026-03-18", "Delivered"),
        (1007, "C01", "2026-04-01", "Shipped"),
        (1008, "C06", "2026-04-03", "Pending")
    ]
)
cursor.executemany(
    "INSERT INTO order_items (order_id, product_id, qty, unit_price) VALUES (%s, %s, %s, %s)",
    [
        (1001, "P1", 2, 799),
        (1001, "P3", 5, 120),
        (1002, "P2", 1, 1999),
        (1003, "P4", 2, 250),
        (1003, "P6", 1, 499),
        (1004, "P5", 1, 1499),
        (1005, "P1", 1, 799),
        (1005, "P2", 1, 1999),
        (1005, "P3", 10, 120),
        (1006, "P6", 3, 499),
        (1007, "P5", 1, 1499),
        (1008, "P4", 4, 250)
    ]
)
cursor.executemany(
    "INSERT INTO payments (payment_id, order_id, payment_date, amount, method) VALUES (%s, %s, %s, %s, %s)",
    [
        ("PM01", 1001, "2026-03-02", 2198, "UPI"),
        ("PM02", 1002, "2026-03-03", 1000, "Card"),
        ("PM03", 1002, "2026-03-05", 999, "Card"),
        ("PM04", 1003, "2026-03-10", 500, "UPI"),
        ("PM05", 1005, "2026-03-15", 3998, "NetBanking"),
        ("PM06", 1006, "2026-03-18", 1497, "UPI"),
        ("PM07", 1009, "2026-04-02", 750, "UPI"),
        ("PM08", 1004, "2026-03-12", 1499, "Card")
    ]
)
connection.commit()
show("""
WITH item_totals AS (
    SELECT
        order_id,
        SUM(qty) AS item_count,
        SUM(qty * unit_price) AS order_value
    FROM order_items
    GROUP BY order_id
),
payment_totals AS (
    SELECT order_id, SUM(amount) AS paid_amount
    FROM payments
    GROUP BY order_id
)
SELECT
    o.order_id,
    c.customer_name,
    o.order_date,
    o.status,
    COALESCE(i.item_count, 0) AS item_count,
    COALESCE(i.order_value, 0) AS order_value,
    COALESCE(p.paid_amount, 0) AS paid_amount,
    COALESCE(i.order_value, 0) - COALESCE(p.paid_amount, 0) AS balance,
    CASE
        WHEN COALESCE(p.paid_amount, 0) >= COALESCE(i.order_value, 0) THEN 'Paid'
        WHEN COALESCE(p.paid_amount, 0) > 0 THEN 'Partially Paid'
        ELSE 'Unpaid'
    END AS payment_status
FROM orders o
INNER JOIN customers c ON c.customer_id = o.customer_id
LEFT JOIN item_totals i ON i.order_id = o.order_id
LEFT JOIN payment_totals p ON p.order_id = o.order_id
ORDER BY o.order_id
""")


,order_id,customer_name,order_date,status,item_count,order_value,paid_amount,balance,payment_status
0,1001,Rahul Mehta,2026-03-02,Delivered,7,2198.00,2198.00,0.00,Paid
1,1002,Priya Nair,2026-03-03,Delivered,1,1999.00,1999.00,0.00,Paid
2,1003,Rahul Mehta,2026-03-10,Delivered,3,999.00,500.00,499.00,Partially Paid
3,1004,Arjun Das,2026-03-12,Cancelled,1,1499.00,1499.00,0.00,Paid
4,1005,Fatima Sheikh,2026-03-15,Shipped,12,3998.00,3998.00,0.00,Paid
5,1006,Karan Malhotra,2026-03-18,Delivered,3,1497.00,1497.00,0.00,Paid
6,1007,Rahul Mehta,2026-04-01,Shipped,1,1499.00,0.00,1499.00,Unpaid
7,1008,Sneha Kulkarni,2026-04-03,Pending,4,1000.00,0.00,1000.00,Unpaid


In [12]:
show("""
WITH order_values AS (
    SELECT
        o.order_id,
        o.status AS order_status,
        COALESCE(SUM(oi.qty * oi.unit_price), 0) AS order_value
    FROM orders o
    LEFT JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.status
),
payment_totals AS (
    SELECT
        order_id,
        SUM(amount) AS paid_amount,
        COUNT(*) AS payment_count
    FROM payments
    GROUP BY order_id
),
full_joined AS (
    SELECT
        ov.order_id,
        ov.order_status,
        ov.order_value,
        COALESCE(pt.paid_amount, 0) AS paid_amount,
        COALESCE(pt.payment_count, 0) AS payment_count,
        1 AS order_exists
    FROM order_values ov
    LEFT JOIN payment_totals pt ON pt.order_id = ov.order_id
    UNION ALL
    SELECT
        pt.order_id,
        NULL AS order_status,
        NULL AS order_value,
        pt.paid_amount,
        pt.payment_count,
        0 AS order_exists
    FROM payment_totals pt
    LEFT JOIN order_values ov ON ov.order_id = pt.order_id
    WHERE ov.order_id IS NULL
)
SELECT
    order_id,
    order_status,
    order_value,
    paid_amount,
    CASE
        WHEN order_exists = 0 THEN 'Unknown Order'
        WHEN order_status = 'Cancelled' AND payment_count > 0 THEN 'Refund Required'
        WHEN order_status <> 'Cancelled' AND payment_count = 0 THEN 'Payment Missing'
        WHEN paid_amount < order_value THEN 'Short Paid'
    END AS recon_status
FROM full_joined
WHERE order_exists = 0
   OR (order_status = 'Cancelled' AND payment_count > 0)
   OR (order_status <> 'Cancelled' AND payment_count = 0)
   OR (order_status <> 'Cancelled' AND payment_count > 0 AND paid_amount < order_value)
ORDER BY order_id
""")


,order_id,order_status,order_value,paid_amount,recon_status
0,1003,Delivered,999.00,500.00,Short Paid
1,1004,Cancelled,1499.00,1499.00,Refund Required
2,1007,Shipped,1499.00,0.00,Payment Missing
3,1008,Pending,1000.00,0.00,Payment Missing
4,1009,None,None,750.00,Unknown Order


In [13]:
cursor.execute("DROP TABLE IF EXISTS employees")
cursor.execute("DROP TABLE IF EXISTS departments")
cursor.execute("""
CREATE TABLE departments (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(60) NOT NULL,
    location VARCHAR(50)
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE employees (
    emp_id VARCHAR(5) PRIMARY KEY,
    emp_name VARCHAR(60) NOT NULL,
    dept_id INT,
    manager_id VARCHAR(5),
    salary DECIMAL(12, 2) NOT NULL,
    hire_date DATE NOT NULL
) ENGINE=InnoDB
""")
cursor.executemany(
    "INSERT INTO departments (dept_id, dept_name, location) VALUES (%s, %s, %s)",
    [
        (10, "Leadership", "Jaipur"),
        (20, "Engineering", "Bengaluru"),
        (30, "Analytics", "Pune"),
        (40, "Finance", "Mumbai"),
        (50, "Legal", "Delhi")
    ]
)
cursor.executemany(
    "INSERT INTO employees (emp_id, emp_name, dept_id, manager_id, salary, hire_date) VALUES (%s, %s, %s, %s, %s, %s)",
    [
        ("E01", "Asha Verma", 10, None, 150000, "2018-04-01"),
        ("E02", "Ravi Kumar", 20, "E01", 95000, "2019-06-15"),
        ("E03", "Meena Pillai", 20, "E02", 72000, "2021-01-10"),
        ("E04", "John Mathew", 20, "E02", 72000, "2022-03-01"),
        ("E05", "Sara Ali", 30, "E01", 98000, "2019-09-20"),
        ("E06", "Vivek Rao", 30, "E05", 64000, "2023-02-14"),
        ("E07", "Pooja Shah", 30, "E05", 58000, "2024-07-01"),
        ("E08", "Nikhil Jain", 20, "E03", 55000, "2025-01-06"),
        ("E09", "Tara Menon", 40, "E01", 88000, "2020-11-11"),
        ("E10", "Imran Qureshi", None, "E01", 61000, "2025-08-18")
    ]
)
connection.commit()
show("""
SELECT
    e.emp_id,
    e.emp_name,
    COALESCE(m.emp_name, 'No Manager') AS manager_name,
    COALESCE(d.dept_name, 'Not Assigned') AS dept_name,
    COALESCE(d.location, 'Not Assigned') AS location
FROM employees e
LEFT JOIN employees m ON m.emp_id = e.manager_id
LEFT JOIN departments d ON d.dept_id = e.dept_id
ORDER BY e.emp_id
""")
show("""
SELECT d.dept_id, d.dept_name, d.location
FROM employees e
RIGHT JOIN departments d ON d.dept_id = e.dept_id
WHERE e.emp_id IS NULL
ORDER BY d.dept_id
""")


,emp_id,emp_name,manager_name,dept_name,location
0,E01,Asha Verma,No Manager,Leadership,Jaipur
1,E02,Ravi Kumar,Asha Verma,Engineering,Bengaluru
2,E03,Meena Pillai,Ravi Kumar,Engineering,Bengaluru
3,E04,John Mathew,Ravi Kumar,Engineering,Bengaluru
4,E05,Sara Ali,Asha Verma,Analytics,Pune
5,E06,Vivek Rao,Sara Ali,Analytics,Pune
6,E07,Pooja Shah,Sara Ali,Analytics,Pune
7,E08,Nikhil Jain,Meena Pillai,Engineering,Bengaluru
8,E09,Tara Menon,Asha Verma,Finance,Mumbai
9,E10,Imran Qureshi,Asha Verma,Not Assigned,Not Assigned


,dept_id,dept_name,location
0,50,Legal,Delhi


In [14]:
show("""
SELECT (
    SELECT DISTINCT salary
    FROM employees
    ORDER BY salary DESC
    LIMIT 1 OFFSET 3
) AS fourth_highest_salary
""")
show("""
SELECT
    e.emp_name,
    e.dept_id,
    e.salary,
    ROUND(d.dept_avg_salary, 2) AS dept_avg_salary
FROM employees e
INNER JOIN (
    SELECT dept_id, AVG(salary) AS dept_avg_salary
    FROM employees
    WHERE dept_id IS NOT NULL
    GROUP BY dept_id
) d ON d.dept_id = e.dept_id
WHERE e.salary > d.dept_avg_salary
ORDER BY e.dept_id, e.salary DESC
""")
show("""
SELECT
    e.emp_id,
    e.emp_name,
    (
        SELECT COUNT(*)
        FROM employees r
        WHERE r.manager_id = e.emp_id
    ) AS direct_reports
FROM employees e
WHERE EXISTS (
    SELECT 1
    FROM employees r
    WHERE r.manager_id = e.emp_id
)
ORDER BY e.emp_id
""")


,fourth_highest_salary
0,88000.00


,emp_name,dept_id,salary,dept_avg_salary
0,Ravi Kumar,20,95000.00,73500.00
1,Sara Ali,30,98000.00,73333.33


,emp_id,emp_name,direct_reports
0,E01,Asha Verma,4
1,E02,Ravi Kumar,2
2,E03,Meena Pillai,1
3,E05,Sara Ali,2


In [15]:
cursor.execute("DROP TABLE IF EXISTS crm_customers")
cursor.execute("DROP TABLE IF EXISTS billing_customers")
cursor.execute("""
CREATE TABLE crm_customers (
    customer_id VARCHAR(5) NOT NULL,
    email VARCHAR(100) NOT NULL
) ENGINE=InnoDB
""")
cursor.execute("""
CREATE TABLE billing_customers (
    customer_id VARCHAR(5) NOT NULL,
    email VARCHAR(100) NOT NULL
) ENGINE=InnoDB
""")
cursor.executemany(
    "INSERT INTO crm_customers (customer_id, email) VALUES (%s, %s)",
    [
        ("C01", "rahul.mehta@mail.com"),
        ("C02", "priya.nair@mail.com"),
        ("C02", "priya.nair@mail.com"),
        ("C03", "arjun.das@mail.com"),
        ("C04", "fatima.s@mail.com")
    ]
)
cursor.executemany(
    "INSERT INTO billing_customers (customer_id, email) VALUES (%s, %s)",
    [
        ("C02", "priya.nair@mail.com"),
        ("C04", "fatima.s@mail.com"),
        ("C05", "karan.m@mail.com"),
        ("C06", "sneha.k@mail.com")
    ]
)
connection.commit()
show("""
SELECT customer_id, email
FROM crm_customers
INTERSECT
SELECT customer_id, email
FROM billing_customers
ORDER BY customer_id
""")
show("""
SELECT customer_id, email
FROM crm_customers
EXCEPT
SELECT customer_id, email
FROM billing_customers
ORDER BY customer_id
""")
show("""
SELECT customer_id, email
FROM billing_customers
EXCEPT
SELECT customer_id, email
FROM crm_customers
ORDER BY customer_id
""")
show("""
SELECT
    (SELECT COUNT(*) FROM (
        SELECT customer_id, email FROM crm_customers
        UNION
        SELECT customer_id, email FROM billing_customers
    ) AS union_result) AS union_rows,
    (SELECT COUNT(*) FROM (
        SELECT customer_id, email FROM crm_customers
        UNION ALL
        SELECT customer_id, email FROM billing_customers
    ) AS union_all_result) AS union_all_rows
""")


,customer_id,email
0,C02,priya.nair@mail.com
1,C04,fatima.s@mail.com


,customer_id,email
0,C01,rahul.mehta@mail.com
1,C03,arjun.das@mail.com


,customer_id,email
0,C05,karan.m@mail.com
1,C06,sneha.k@mail.com


,union_rows,union_all_rows
0,6,9


In [16]:
connection.close()
